# RGB-Agent Local Evaluation

This notebook is for running `RGB-Agent` locally on your own machine.

It supports:
- installing this project in editable mode
- loading a local Hugging Face model directly with `transformers`
- running `RGB-Agent` offline evaluation against local `environment_files/`
- inspecting `summary.txt`, `summary.csv`, and `scorecard.json`

Recommended local setup for your machine:
- GPU: `RTX PRO 6000 Blackwell 96GB`
- Model: `Qwen/Qwen2.5-72B-Instruct` or another local checkpoint path
- Backend: `transformers`
- Analyzer backend: `transformers`


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists() or not (PROJECT_ROOT / 'rgb_agent').exists():
    raise FileNotFoundError('Run this notebook from the RGB-Agent project root.')

ENV_DIR = PROJECT_ROOT / 'environment_files'
OUTPUT_ROOT = PROJECT_ROOT / 'evaluation_results'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('ENV_DIR      =', ENV_DIR)
print('OUTPUT_ROOT  =', OUTPUT_ROOT)


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'bitsandbytes'])
print('Project installed in editable mode.')


## Configuration

Edit this cell before running.

Typical local use:
- point `MODEL_DIR` at your downloaded model directory
- set `MODEL_DIR` to your downloaded local model directory
- default settings use 4-bit loading through `bitsandbytes`
- keep the analyzer prompt compact to avoid context overflow


In [ ]:
AGENT_NAME = 'rgb_agent'
SUITE = 'all'
GAME = None
MAX_ACTIONS = 500
ANALYZER_INTERVAL = 10
ANALYZER_RETRIES = 5
DESCRIPTION = 'local-transformers-run'

MODEL_DIR = Path('/absolute/path/to/Qwen2.5-72B-Instruct')

ANALYZER_BACKEND = 'transformers'
MODEL = str(MODEL_DIR)

# Conservative analyzer limits to reduce prompt/context overflow.
ANALYZER_HEAD_CHARS = '1000'
ANALYZER_TAIL_CHARS = '4000'
ANALYZER_RECENT_BLOCKS = '4'
ANALYZER_MAX_OUTPUT_TOKENS = '256'
TRANSFORMERS_LOAD_IN_4BIT = '1'
TRANSFORMERS_TORCH_DTYPE = 'bfloat16'

os.environ['ENVIRONMENTS_DIR'] = str(ENV_DIR)
os.environ['ANALYZER_BACKEND'] = ANALYZER_BACKEND
os.environ['LOCAL_TRANSFORMERS_MODEL_PATH'] = str(MODEL_DIR)
os.environ['ANALYZER_HEAD_CHARS'] = ANALYZER_HEAD_CHARS
os.environ['ANALYZER_TAIL_CHARS'] = ANALYZER_TAIL_CHARS
os.environ['ANALYZER_RECENT_BLOCKS'] = ANALYZER_RECENT_BLOCKS
os.environ['ANALYZER_MAX_OUTPUT_TOKENS'] = ANALYZER_MAX_OUTPUT_TOKENS
os.environ['TRANSFORMERS_LOAD_IN_4BIT'] = TRANSFORMERS_LOAD_IN_4BIT
os.environ['TRANSFORMERS_TORCH_DTYPE'] = TRANSFORMERS_TORCH_DTYPE

print('MODEL_DIR        =', MODEL_DIR)
print('MODEL            =', MODEL)
print('ANALYZER_BACKEND =', ANALYZER_BACKEND)
print('HEAD_CHARS       =', ANALYZER_HEAD_CHARS)
print('TAIL_CHARS       =', ANALYZER_TAIL_CHARS)
print('RECENT_BLOCKS    =', ANALYZER_RECENT_BLOCKS)
print('MAX_OUTPUT_TOKENS=', ANALYZER_MAX_OUTPUT_TOKENS)
print('LOAD_IN_4BIT     =', TRANSFORMERS_LOAD_IN_4BIT)


In [ ]:
import importlib

if not MODEL_DIR.exists():
    raise FileNotFoundError(f'Model directory not found: {MODEL_DIR}')
print('Using local transformers model path:', MODEL_DIR)

# Reload project modules after env vars are set so analyzer limits are applied.
import rgb_agent.agent.direct_analyzer
import rgb_agent.agent.local_transformers_analyzer
import rgb_agent.environment.runner
import rgb_agent.evaluation.offline_eval

importlib.reload(rgb_agent.agent.direct_analyzer)
importlib.reload(rgb_agent.agent.local_transformers_analyzer)
importlib.reload(rgb_agent.environment.runner)
importlib.reload(rgb_agent.evaluation.offline_eval)
print('RGB-Agent modules reloaded with current analyzer env settings.')


In [ ]:
from rgb_agent.evaluation.offline_eval import run_offline_evaluation

result = run_offline_evaluation(
    agent_name=AGENT_NAME,
    game=GAME,
    suite=SUITE,
    max_actions=MAX_ACTIONS,
    analyzer_interval=ANALYZER_INTERVAL,
    analyzer_model=MODEL,
    analyzer_backend=ANALYZER_BACKEND,
    analyzer_retries=ANALYZER_RETRIES,
    description=DESCRIPTION,
    output_root=OUTPUT_ROOT,
    environments_dir=ENV_DIR,
)

result


In [ ]:
from pathlib import Path
import json

run_dir = Path(result['run_dir'])
print('Run dir:', run_dir)
print('\nArtifacts:')
for path in sorted(run_dir.iterdir()):
    print('-', path.name)

if result.get('scorecard_json'):
    scorecard = json.loads(Path(result['scorecard_json']).read_text())
    print('\nOverall score:', scorecard.get('score'))


In [ ]:
summary_path = Path(result['summary_txt'])
print(summary_path.read_text())


In [ ]:
import pandas as pd

csv_path = Path(result['summary_csv'])
df = pd.read_csv(csv_path)
df
